# [Azure OpenAI Responses API's](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/responses?tabs=python-secure#generate-a-text-response) (with Entra ID authentication)

In [1]:
import os, sys
from openai import AzureOpenAI
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential, get_bearer_token_provider

# load ".env" from the kernel working folder, e.g. same folder where we ran "Jupyter notebook"
if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

openai_api_version    = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)

LANGUAGE = "French" # "English", "Italian", "French", "Japanese"...

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

client = AzureOpenAI(
    azure_ad_token_provider=token_provider,
    api_version=openai_api_version,
    azure_endpoint=azure_openai_endpoint
)

print(f"openai_endpoint: {azure_openai_endpoint}")
print(f"azure_deployment_name: {azure_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
azure_deployment_name: gpt-5.4-mini
openai_api_version: 2025-04-01-preview


# Read the system message plus the json list of metrics

In [2]:
import json

# Open the system_message file in read mode
# The file lives in the "notebook working folder" where the physical file .ipynb is stored
with open("system_message.txt", "r", encoding="utf-8") as file:
    system_message = file.read()

# Open the metrics file in read mode
with open("metrics.json", "r", encoding="utf-8") as file:
    metrics_list = json.loads(file.read().replace("\n", ""))

i=1
for m in metrics_list:
    print (f'{i}: {m["metric_name"]}')
    i += 1

1: Intent Resolution
2: Tool Call Accuracy
3: Task Adherence
4: Response Completeness
5: Groundedness (prompt-based)
6: Groundedness Pro
7: Retrieval
8: Relevance
9: Coherence
10: Fluency
11: Similarity
12: F1 Score
13: BLEU Score
14: ROUGE Score
15: METEOR Score


# You may run the next cell multiple times to explore metrics samples

In [3]:
from IPython.display import Markdown, display

index = int(input("Which metric would you like to analyze?"))

display(Markdown(f"\n\n**GREAT CHOICE!** Here are a couple examples for the metric `<{metrics_list[index-1]['metric_name']}>`\n"))

messages=[{"role": "system", "content": f"{system_message}.\nQUESTION, ANSWER, SCORE AND  EXPLANATION must be in {LANGUAGE}"},
          {"role": "user", "content": json.dumps(metrics_list[index-1])}]

response = client.responses.create(
    model=os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'],
    input=messages)

display(Markdown(response.output_text.replace("\n", "  \n")))

Which metric would you like to analyze? 11




**GREAT CHOICE!** Here are a couple examples for the metric `<Similarity>`


Voici des exemples **spécifiques au métrique Similarity** (similarité entre la réponse générée et la vérité terrain, **par rapport à une requête**), avec une échelle **de 1 à 5**.  
  
---  
  
## Score 5 — Similarité quasi parfaite  
**QUESTION :** Résume en une phrase la révolution française.    
**ANSWER :** La Révolution française a renversé l’Ancien Régime et instauré de nouveaux principes politiques fondés sur la liberté et l’égalité.    
**SCORE :** 5    
**EXPLICATION :** La réponse reprend presque exactement l’idée attendue de la vérité terrain : mêmes concepts clés, même sens, aucune divergence importante.  
  
---  
  
## Score 4 — Très proche, légère différence de formulation  
**QUESTION :** Résume en une phrase la révolution française.    
**ANSWER :** La Révolution française a mis fin à la monarchie absolue et a transformé durablement la société française.    
**SCORE :** 4    
**EXPLICATION :** La réponse est très similaire à la référence, mais elle est un peu moins complète et n’exprime pas tous les éléments attendus.  
  
---  
  
## Score 3 — Similarité moyenne  
**QUESTION :** Résume en une phrase la révolution française.    
**ANSWER :** La Révolution française a été une période de grands changements politiques en Europe.    
**SCORE :** 3    
**EXPLICATION :** La réponse est liée au sujet, mais elle est plus générale et s’éloigne partiellement de la réponse de référence.  
  
---  
  
## Score 2 — Faible similarité  
**QUESTION :** Résume en une phrase la révolution française.    
**ANSWER :** La Révolution française a surtout concerné les guerres entre plusieurs pays européens au XVIIIe siècle.    
**SCORE :** 2    
**EXPLICATION :** Il y a un lien faible avec le thème, mais plusieurs éléments sont imprécis ou incorrects par rapport à la vérité terrain.  
  
---  
  
## Score 1 — Très faible similarité  
**QUESTION :** Résume en une phrase la révolution française.    
**ANSWER :** La Révolution française est un roman sur un jeune explorateur dans le désert.    
**SCORE :** 1    
**EXPLICATION :** La réponse n’a pratiquement aucun rapport avec la réponse attendue ni avec la requête.  
  
---  
  
Si vous voulez, je peux maintenant faire la même chose pour les 14 autres métriques, **une par une**, dans le même format pédagogique.